In [6]:
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

import ecgformer

In [7]:
# beats_train_x = np.load('/home/matrioszka/ptb-xl/X_train.npy')
# beats_train_y = np.load('/home/matrioszka/ptb-xl/y_train.npy', allow_pickle=True)
# beats_test_x = np.load('/home/matrioszka/ptb-xl/X_test.npy')
# beats_test_y = np.load('/home/matrioszka/ptb-xl/y_test.npy', allow_pickle=True)
beats_x = np.load('/home/matrioszka/mit-bih/mitbih_beats_x.npy', allow_pickle=True)
beats_y = np.load('/home/matrioszka/mit-bih/mitbih_beats_y.npy', allow_pickle=True)
print(beats_x.shape, beats_y.shape)

(85243, 150) (85243,)


In [8]:
beats_y = np.array(['N' if label=='N' else 'O' for label in beats_y])
# split into train/test
from sklearn.model_selection import train_test_split
beats_train_x, beats_test_x, beats_train_y, beats_test_y = train_test_split(
    beats_x, beats_y, test_size=0.2, random_state=45, stratify=beats_y)

le = LabelEncoder()
y_train = le.fit_transform(beats_train_y)
y_test  = le.transform(beats_test_y)

print(le.classes_)

X_train_t = torch.tensor(beats_train_x, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(beats_test_x, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

class_counts = np.bincount(y_train)  # assumes integer-encoded labels 0–4
print(class_counts)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()  # normalize to sum to 1
print(class_weights)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to("cuda")
criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

['N' 'O']
[59366  8828]
[0.1294542 0.8705458]


In [ ]:
model = ecgformer.ECGMatformer(
    input_length=150,
    patch_size=10,
    d_model=64,
    num_heads=4,
    num_layers=4,
    d_ffn=64,
    dropout=0.15,
    num_classes=2,
    device="cuda",
    matryoshka_depth=4,
)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ECGformer | trainable params: {total:,}")
model = model.to("cuda")
optimizer = ecgformer.build_optimizer(model)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=1e-3,
        steps_per_epoch=len(train_loader),
        epochs=100,
        pct_start=0.1,
    )

best_val_loss = float("inf")
epochs = 100

for epoch in range(1, epochs):
    train_loss = ecgformer.train_one_epoch(model, train_loader, optimizer, scheduler, "cuda", criterion)
    val_metrics = ecgformer.evaluate(model, test_loader, "cuda", criterion)
    val_loss, val_acc = val_metrics["loss"], val_metrics["accuracy"]

    scheduler.step(val_loss)

    print(
        f"Epoch [{epoch:3d}/{epochs}] "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}  "
        f"val_acc={val_acc*100:.2f}%"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "ecgformer_best_all.pth")
        print(f"  ✓ Saved best model → ecgformer_best_all.pth")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

ECGformer | trainable params: 101,826
{'matryoshka_granularity': 3, 'loss': 0.6927169928334522, 'accuracy': 0.8705495923514576}
{'matryoshka_granularity': 2, 'loss': 0.6850795025657409, 'accuracy': 0.8466772244706434}
{'matryoshka_granularity': 1, 'loss': 0.47647453603929657, 'accuracy': 0.6992198955950496}
{'matryoshka_granularity': 0, 'loss': 0.47952302669987973, 'accuracy': 0.7296029092615403}
Epoch [  1/100] train_loss=0.7882  val_loss=0.4795  val_acc=72.96%
  ✓ Saved best model → ecgformer_best_all.pth


/tmp/ipykernel_464745/2978436823.py:32: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(val_loss)


{'matryoshka_granularity': 3, 'loss': 0.6902948864408724, 'accuracy': 0.8668543609595871}
{'matryoshka_granularity': 2, 'loss': 0.441464796011186, 'accuracy': 0.8090797114200247}
{'matryoshka_granularity': 1, 'loss': 0.4562644064496344, 'accuracy': 0.7782274620212329}
{'matryoshka_granularity': 0, 'loss': 0.4797235629070371, 'accuracy': 0.7602791952607191}
Epoch [  2/100] train_loss=0.5983  val_loss=0.4797  val_acc=76.03%
{'matryoshka_granularity': 3, 'loss': 0.4047769697920459, 'accuracy': 0.9064461258724852}
{'matryoshka_granularity': 2, 'loss': 0.3774380725607144, 'accuracy': 0.910727901929732}
{'matryoshka_granularity': 1, 'loss': 0.39253081658647393, 'accuracy': 0.879758343597865}
{'matryoshka_granularity': 0, 'loss': 0.37010398075002904, 'accuracy': 0.8927209807026805}
Epoch [  3/100] train_loss=0.4869  val_loss=0.3701  val_acc=89.27%
  ✓ Saved best model → ecgformer_best_all.pth
{'matryoshka_granularity': 3, 'loss': 0.3596405790914111, 'accuracy': 0.9150683324535164}
{'matryoshk

In [5]:
# load the model and evaluate on test set precision, recall, f1-score for each class
model.load_state_dict(torch.load("ecgformer_best_all.pth"))
model.eval()
from sklearn.metrics import classification_report

matryoshka_granularities = [i for i in range(model.matryoshka_depth)][::-1]
with torch.no_grad():
    for g in matryoshka_granularities:
        all_preds = []
        all_labels = []
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to("cuda"), y_batch.to("cuda")
            outputs = model(X_batch, g)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
        print(f"\nMatryoshka Granularity: {g}")
        print(classification_report(all_labels, all_preds, target_names=le.classes_))


Matryoshka Granularity: 3
              precision    recall  f1-score   support

           N       0.99      0.96      0.97     14842
           O       0.77      0.92      0.84      2207

    accuracy                           0.95     17049
   macro avg       0.88      0.94      0.91     17049
weighted avg       0.96      0.95      0.96     17049


Matryoshka Granularity: 2
              precision    recall  f1-score   support

           N       0.99      0.97      0.98     14842
           O       0.80      0.92      0.86      2207

    accuracy                           0.96     17049
   macro avg       0.89      0.94      0.92     17049
weighted avg       0.96      0.96      0.96     17049


Matryoshka Granularity: 1
              precision    recall  f1-score   support

           N       0.99      0.96      0.98     14842
           O       0.78      0.93      0.85      2207

    accuracy                           0.96     17049
   macro avg       0.89      0.95      0.91    